# Social Media Usage and Emotional Well-Being - Data Analysis and Modeling

## Introduction

Social media has become an integral part of modern life, influencing various aspects of our daily routines and emotional well-being. This analysis explores a dataset that captures social media usage and users' dominant emotional states. The goal of this investigation is to understand the relationship between social media usage patterns and emotional well-being, and to build a predictive model using a GRU (Gated Recurrent Unit) neural network that can predict the dominant emotion of a user based on their social media activity.

We will proceed through the following steps in this notebook:

1. Loading Required Libraries
2. Load the Data
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Building the GRU Neural Network
6. Model Evaluation
7. Conclusion

Let's get started!

## 1. Loading Required Libraries

In this section, we will load the necessary libraries required for data manipulation, visualization, and building our GRU (Gated Recurrent Unit) neural network. We will be using popular Python libraries such as Pandas, NumPy, Matplotlib, Seaborn for handling and visualizing data, as well as TensorFlow for building our machine learning model.

Let's start by importing these libraries.

In [ ]:
# Importing necessary libraries for data manipulation
import pandas as pd  # Powerful data structures for data analysis, time series, and statistics
import numpy as np  # Fundamental package for scientific computing with Python

# Importing libraries for data visualization
import seaborn as sns  # Statistical data visualization
import matplotlib.pyplot as plt  # Plotting library for Python

# Importing machine learning utilities
from sklearn.model_selection import train_test_split  # Utility to split the dataset into training and testing sets
from sklearn.preprocessing import StandardScaler  # Standardize features by removing the mean and scaling to unit variance
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix  # Metrics to evaluate the performance of a model
from sklearn.preprocessing import LabelEncoder # Label Encoder

# Importing TensorFlow libraries for building the GRU model
import tensorflow as tf  # Deep learning library
from tensorflow.keras.models import Sequential  # Sequential model type from Keras
from tensorflow.keras.layers import Dense, GRU, Dropout  # Layers used to build the neural network
from tensorflow.keras.utils import to_categorical # Categorical of data

# Setting visualization styles
sns.set(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)  # Setting default figure size for plots

# Fix the seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Print TensorFlow version
print(f'TensorFlow version: {tf.__version__}')


### Explanation:

1. **Pandas (`pd`)**: Used for data manipulation and analysis. It provides data structures and functions needed to manipulate structured data effortlessly.
2. **NumPy (`np`)**: Fundamental package for scientific computing with Python, providing comprehensive mathematical functions and array handling.
3. **Seaborn (`sns`)**: A library for making statistical graphics in Python. It is built on top of Matplotlib and closely integrated with Pandas data structures.
4. **Matplotlib (`plt`)**: A plotting library used for 2D graphics. It is very useful for visualizing data and creating plots.
5. **Scikit-learn**:
   - `train_test_split`: Used to split the dataset into training and testing sets.
   - `StandardScaler`: Used for feature scaling, ensuring standardized value ranges for models.
   - `classification_report`, `accuracy_score`, `confusion_matrix`: Provide metrics to assess model performance.
6. **TensorFlow (`tf`)**: An end-to-end open-source platform for machine learning.
   - `Sequential`: Model class that allows us to build models layer by layer.
   - `Dense`, `GRU`, `Dropout`: Different layers we'll use in our deep learning model.
7. **Seeding for Reproducibility**: Setting the random seed ensures that the random operations are reproducible which is useful for debugging.

In this section, we've prepared our workspace by loading the necessary libraries. Next, we will proceed to load and explore the dataset in subsection 2.

## 2. Load the Data
### 2.1 Loading the Data

Now, we will load our dataset files (`train.csv`, `test.csv`, and `val.csv`) into Pandas DataFrames. This is the first step in any data analysis workflow, and it allows us to begin exploring and understanding our data.

In [ ]:
# Load the datasets
train_df = pd.read_csv('/kaggle/input/social-media-usage-and-emotional-well-being/train.csv', on_bad_lines='skip')
test_df = pd.read_csv('/kaggle/input/social-media-usage-and-emotional-well-being/test.csv', on_bad_lines='skip')
val_df = pd.read_csv('/kaggle/input/social-media-usage-and-emotional-well-being/val.csv', on_bad_lines='skip')

# Display the first few rows of the training data
print("First 5 rows of the training data:")
train_df.head()

### 2.2 Understanding the Data

Let's explore the basic structure of the DataFrames to understand the data we are working with. We will look at data types, summary statistics, and check for any missing values.

In [ ]:
# Display the structure of the training data
print("Information about the training data:")
train_df.info()

# Display summary statistics
print("\nSummary statistics of the training data:")
train_df.describe()

# Check for missing values
print("\nMissing values in the training data:")
train_df.isnull().sum()

### 2.3 Preview the Test and Validation Data

Let's also take a quick look at the test and validation datasets to ensure they have similar structures to the training dataset.

In [ ]:
# Display the first few rows of the test data
print("First 5 rows of the test data:")
test_df.head()

# Check for missing values in the test data
print("\nMissing values in the test data:")
test_df.isnull().sum()

# Display the first few rows of the validation data
print("First 5 rows of the validation data:")
val_df.head()

# Check for missing values in the validation data
print("\nMissing values in the validation data:")
val_df.isnull().sum()

### Conclusion of Step 2

We have successfully loaded our datasets and obtained an overview of their structure. The next step will involve conducting an exploratory data analysis (EDA) to uncover patterns and insights within the data.

# 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) helps us understand the data better by summarizing its main characteristics, often using visual methods. This section will cover various aspects of our dataset to uncover patterns, spot anomalies, and check assumptions.

## 3.1 Data Overview

Let's start by getting an overview of our datasets. We will look at the head of the DataFrame to see the first few rows and understand the structure comprehensively. 

In [ ]:
# Display the first few rows of each dataset
print("First 5 rows of the training data:")
display(train_df.head())

print("\nFirst 5 rows of the test data:")
display(test_df.head())

print("\nFirst 5 rows of the validation data:")
display(val_df.head())

## 3.2 Distribution of Age

Next, we will examine the age distribution of users in our dataset. Understanding age distribution is crucial as it offers insights into the demographic characteristics of the users.

In [ ]:
# Plot age distribution
plt.figure(figsize=(10, 6))
sns.histplot(train_df['Age'], kde=True, bins=30)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

## 3.3 Gender Distribution

Let's visualize the distribution of genders in our training set. This will help us understand the gender diversity within our dataset.

In [ ]:
# Plot gender distribution
plt.figure(figsize=(7, 5))
sns.countplot(x='Gender', data=train_df)
plt.title('Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

## 3.4 Daily Usage Time by Platform

Analyzing how much time users spend on different social media platforms can reveal behavioral patterns. We will visualize the average daily usage time for each platform.

In [ ]:
# Plot average daily usage time by platform
plt.figure(figsize=(12, 6))
sns.barplot(x='Platform', y='Daily_Usage_Time (minutes)', data=train_df, estimator=np.mean)
plt.title('Average Daily Usage Time by Platform')
plt.xlabel('Platform')
plt.ylabel('Daily Usage Time (minutes)')
plt.show()

## 3.5 Dominant Emotion Distribution

Finally, let's look at the distribution of dominant emotions among users. This will give us an idea of the most prevalent emotional states represented in our dataset.

In [ ]:
# Plot dominant emotion distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='Dominant_Emotion', data=train_df)
plt.title('Dominant Emotion Distribution')
plt.xlabel('Dominant Emotion')
plt.ylabel('Count')
plt.show()

### Conclusion of Step 3

Through Exploratory Data Analysis (EDA), we have gained a comprehensive understanding of our data. We explored the age and gender distributions, daily usage time across different platforms, and the distribution of dominant emotions. These insights will guide us through the data preprocessing and modeling phases.

# 4. Data Preprocessing

Data preprocessing involves preparing and transforming raw data before it is used in modeling. This step ensures that the data is clean, well-formatted, and ready for training machine learning models.

## 4.1 Handling Missing Values

We begin by checking for and handling any missing values in the dataset. Missing values can skew the results and affect the performance of machine learning models.

In [ ]:
# Check for missing values in the training dataset
print("Missing values in training data:")
print(train_df.isnull().sum())

# Check for missing values in the test dataset
print("\nMissing values in test data:")
print(test_df.isnull().sum())

# Check for missing values in the validation dataset
print("\nMissing values in validation data:")
print(val_df.isnull().sum())

# Handling missing values (if any) - here, we'll just drop rows with missing values for simplicity
# In a real-world scenario, you might want to impute missing values instead
train_df = train_df.dropna()
test_df = test_df.dropna()
val_df = val_df.dropna()

# Verify that there are no more missing values
print("\nMissing values in training data after dropping rows with missing values:")
print(train_df.isnull().sum())

## 4.2 Encoding Categorical Variables

Many machine learning algorithms require numerical input. We need to convert categorical variables such as 'Gender' and 'Platform' into numerical values. This process is called encoding. We'll use one-hot encoding to handle these categorical variables.

In [ ]:
# One-hot encode the categorical variables for train, test, and validation sets
train_df = pd.get_dummies(train_df, columns=['Gender', 'Platform'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['Gender', 'Platform'], drop_first=True)
val_df = pd.get_dummies(val_df, columns=['Gender', 'Platform'], drop_first=True)

# Ensure that the columns match in train, test, and validation sets
# Align the test and validation sets with the training set
missing_cols_test = set(train_df.columns) - set(test_df.columns)
for col in missing_cols_test:
    test_df[col] = 0
test_df = test_df[train_df.columns]

missing_cols_val = set(train_df.columns) - set(val_df.columns)
for col in missing_cols_val:
    val_df[col] = 0
val_df = val_df[train_df.columns]

# Verify the changes
print("Columns in training data:", train_df.columns)
print("\nColumns in test data after encoding and alignment:", test_df.columns)
print("\nColumns in validation data after encoding and alignment:", val_df.columns)

## 4.3 Feature Scaling

Feature scaling is essential for neural networks. It ensures that each feature contributes proportionally to the calculations in algorithms and speeds up convergence in gradient-based algorithms. We'll use `StandardScaler` to scale the features.

In [ ]:
# Ensure that we use consistent label encoding for the target variable across all data splits
label_encoder = LabelEncoder()

# Combine all the labels from train, test, and validation sets to create a unified label encoder
all_labels = list(train_df['Dominant_Emotion']) + list(test_df['Dominant_Emotion']) + list(val_df['Dominant_Emotion'])
label_encoder.fit(all_labels)

# Encode the target variable in each dataset
y_train_encoded = label_encoder.transform(train_df['Dominant_Emotion'])
y_test_encoded = label_encoder.transform(test_df['Dominant_Emotion'])
y_val_encoded = label_encoder.transform(val_df['Dominant_Emotion'])

# Define features and target variable for each dataset
X_train = train_df.drop('Dominant_Emotion', axis=1)
X_test = test_df.drop('Dominant_Emotion', axis=1)
X_val = val_df.drop('Dominant_Emotion', axis=1)

# Verify the feature names
print("Feature names in train data:", X_train.columns.tolist())
print("Feature names in test data:", X_test.columns.tolist())
print("Feature names in validation data:", X_val.columns.tolist())

# Select only numeric columns for scaling
numeric_columns = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Verify the numeric columns
print("Numeric columns in the dataset:", numeric_columns)

# Standardize the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numeric_columns])
X_test_scaled = scaler.transform(X_test[numeric_columns])
X_val_scaled = scaler.transform(X_val[numeric_columns])

# Verify the scaling
print("Scaled feature sample (first 5 rows of the training data):")
print(X_train_scaled[:5])

### Conclusion of Step 4

We have successfully preprocessed our data by handling missing values, encoding categorical variables, and scaling the features. Our data is now ready for building a neural network model. In the next step, we will construct and train a GRU (Gated Recurrent Unit) neural network to predict the dominant emotion of the user.

# 5. Building the GRU Neural Network

In this section, we will build a GRU (Gated Recurrent Unit) neural network using TensorFlow. GRUs are a type of recurrent neural network (RNN) that are well-suited for processing sequential data.

## 5.1 Data Preparation for GRU

Before building the model, let's ensure our data is ready for the GRU model. We will convert our scaled features and encoded targets into suitable formats for TensorFlow.

In [ ]:
# Convert labels to one-hot encoding
y_train_one_hot = to_categorical(y_train_encoded)
y_test_one_hot = to_categorical(y_test_encoded)
y_val_one_hot = to_categorical(y_val_encoded)

# Ensure the input data is in the correct format (3D array for GRU)
X_train_gru = np.expand_dims(X_train_scaled, axis=1)
X_test_gru = np.expand_dims(X_test_scaled, axis=1)
X_val_gru = np.expand_dims(X_val_scaled, axis=1)

print("Shape of X_train for GRU:", X_train_gru.shape)
print("Shape of Y_train for GRU:", y_train_one_hot.shape)

## 5.2 Custom Activation Function

We will define a custom activation function that will be used in the GRU layers. This function will be tailored specifically for our model and will add a unique characteristic to it.

In [ ]:
# Define a custom and more effective activation function combining multiple properties
@tf.function
def custom_activation(x):
    # Combination of sigmoid, tanh, ReLU, and a linear transformation
    tanh = tf.keras.activations.tanh(x)
    relu = tf.keras.activations.relu(x)
    return tanh + relu * 1.5

# Test the custom activation function
print("Testing custom activation function with sample input:")
sample_input = np.array([0.5, 1.0, 1.5, 2.0])
print(custom_activation(sample_input))

## 5.3 Model Architecture

Let's now define the architecture of our GRU neural network. Our model will consist of multiple GRU layers, followed by dense layers to bring the number of parameters to around 105K.

In [ ]:
# Clear any existing models in the session
tf.keras.backend.clear_session()

# Build the GRU neural network with the updated custom activation function
model = Sequential()

# Adding the first GRU layer with the updated custom activation function
model.add(GRU(32, activation="relu", return_sequences=True, input_shape=(1, X_train_gru.shape[2])))
model.add(Dropout(0.2))

# Adding additional GRU layers
model.add(GRU(128, activation=custom_activation, return_sequences=True))
model.add(Dropout(0.2))

model.add(GRU(64, activation=custom_activation, return_sequences=False))
model.add(Dropout(0.2))

# Adding a dense layer for classification
model.add(Dense(32, activation="tanh"))
model.add(Dropout(0.2))

# Output layer with the number of classes for the target variable
output_classes = y_train_one_hot.shape[1]
model.add(Dense(output_classes, activation='softmax'))

# Summary of the model
model.summary()

# Confirm the total parameters to ensure it's around 2 million
total_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print("Total trainable parameters: ", total_params)

## 5.4 Model Compilation

We need to compile our model using an appropriate optimizer and loss function. We will use 'adam' as the optimizer and 'categorical_crossentropy' as the loss function since we are performing multi-class classification.

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary again to ensure everything is correct
model.summary()

## 5.5 Model Training

Let's train our GRU model using the training data. We will use the validation data to evaluate the model's performance after each epoch.

In [ ]:
# Train the model
history = model.fit(X_train_gru, y_train_one_hot, epochs=300, batch_size=32, validation_data=(X_val_gru, y_val_one_hot))

# Save the model
model.save('gru_emotion_model.h5')

# Plot the training and validation loss and accuracy
plt.figure(figsize=(12, 6))

# Plot training & validation accuracy values
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.show()

### Conclusion of Step 5

We have successfully built, compiled, and trained a GRU neural network for predicting the dominant emotion based on social media usage. The custom activation function and unique model architecture ensure that its parameter count is around 105K. In the next step, we will evaluate the model's performance on the test dataset.

# 6. Model Evaluation

After training our GRU neural network, it is essential to evaluate its performance on unseen data. This helps us understand how well the model generalizes to new data.

We will evaluate our model using the test dataset. The metrics we consider include accuracy, loss, and a confusion matrix to give detailed insights into the model's performance.

## 6.1 Evaluating on Test Set

First, we will evaluate our model on the test dataset using accuracy and loss metrics. This will give us a primary understanding of the model's performance on unseen data.

In [ ]:
# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test_gru, y_test_one_hot)
print(f"\nTest Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

## 6.2 Confusion Matrix and Classification Report

To gain deeper insights into our model's performance, we will create a confusion matrix and a classification report. This will help us understand how well the model is performing for each class.

In [ ]:

# Predict classes for the test set
y_test_pred = model.predict(X_test_gru)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)
y_test_true_classes = np.argmax(y_test_one_hot, axis=1)

# Verify the correct number of classes and target names
unique_labels = np.unique(y_test_true_classes)
unique_label_names = label_encoder.inverse_transform(unique_labels)

# Generate classification report
print("\nClassification Report:")
print(classification_report(y_test_true_classes, y_test_pred_classes, target_names=unique_label_names))

# Generate confusion matrix
conf_matrix = confusion_matrix(y_test_true_classes, y_test_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', xticklabels=unique_label_names, yticklabels=unique_label_names, cmap='Blues')

plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Conclusion of Step 6

We have evaluated the performance of our GRU neural network on the test dataset. The results, including accuracy, loss, and a detailed classification report, provide insights into the strengths and weaknesses of our model.

Our next steps could include further tuning the model, exploring different architectures, and applying techniques to deal with overfitting or underfitting, as indicated by our results.

# 7. Conclusion and Future Work

## 7.1 Summary of Findings

Throughout this notebook, we explored the relationship between social media usage and emotional well-being using a custom-built GRU neural network. Here is a summary of our key findings and outcomes:

- **Exploratory Data Analysis (EDA)**: We examined the distribution of various features like age, gender, daily usage time, and dominant emotions.
- **Data Preprocessing**: We handled missing values, encoded categorical variables, and scaled the features appropriately.
- **Model Building**: We developed a GRU neural network with approximately 2 million parameters using a unique custom activation function tailored to our data.
- **Model Training**: We trained our model on the training data and evaluated it against a validation set. The model showed good training performance, with some overfitting observed.
- **Model Evaluation**: We evaluated the model on a separate test set, deploying both accuracy metrics and detailed classification reports to assess its generalization ability.

Overall, our model demonstrated a satisfactory performance in predicting the dominant emotions based on social media usage data.

## 7.2 Future Work

Based on our findings and observations, here are some potential directions for future work:

1. **Hyperparameter Tuning**: Experiment with different hyperparameters to improve the model's performance and reduce overfitting.
2. **Advanced Architectures**: Explore more advanced neural network architectures such as LSTMs, Transformers, or even hybrid models to better capture the complexities in the data.
3. **Feature Engineering**: Add more features that can enrich the dataset, such as text-based features from social media posts or additional temporal data.
4. **Regularization Techniques**: Apply regularization techniques like dropout, early stopping, and weight decay to improve model generalization.
5. **Cross-Validation**: Use cross-validation techniques to ensure the robustness of the results across different folds of the data.
6. **Ensemble Methods**: Combine multiple models to create an ensemble, which can potentially improve the predictive performance.

## Final Remarks

This notebook provided a comprehensive journey from data exploration to model evaluation, offering valuable insights into the intersection of social media usage and emotional well-being. With further improvements and fine-tuning, the model can become a powerful tool for predicting emotional states based on social media activity.